# Hour 3: Advanced Multi-Capability Assistants
**Combining Everything We've Learned**

## What We're Building:
1. **Enhanced Email Writer** - With web research
2. **Smart Summarizer** - With analytics
3. **Multi-Capability Assistant** - Complete project

**Strategy:** SIMPLE first, then COMPLETE versions!

## Setup

In [ ]:
from openai import OpenAI
import os

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Setup complete!")

✅ Setup complete!


## Tools Setup

In [2]:
# Copy tools
import json

def calculate(expression):
    try:
        return json.dumps({"result": eval(expression)})
    except:
        return json.dumps({"error": "Invalid expression"})

def web_search(query):
    results = {
        "ai trends": "Latest AI: Advanced reasoning, multimodal models",
        "technology": "Tech news: AI adoption accelerating",
        "email tips": "Email tips: Clear subject, concise content"
    }
    for keyword in results:
        if keyword in query.lower():
            return json.dumps({"results": results[keyword]})
    return json.dumps({"results": f"Info about {query}"})

def analyze_data(data_string, operation):
    data = json.loads(data_string)
    if operation == "sum": result = sum(data)
    elif operation == "average": result = sum(data) / len(data)
    elif operation == "max": result = max(data)
    elif operation == "min": result = min(data)
    else: result = None
    return json.dumps({"result": result})

# Tool schemas
calculator_tool = {"type": "function", "function": {"name": "calculate", "description": "Do math", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}}
web_search_tool = {"type": "function", "function": {"name": "web_search", "description": "Search web", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
data_analyzer_tool = {"type": "function", "function": {"name": "analyze_data", "description": "Analyze data", "parameters": {"type": "object", "properties": {"data_string": {"type": "string"}, "operation": {"type": "string", "enum": ["sum", "average", "max", "min"]}}, "required": ["data_string", "operation"]}}}

print("✅ Tools loaded!")

✅ Tools loaded!


---
# Part 1: Enhanced Email Writer (with Research)

## 🎯 SIMPLE VERSION

In [8]:
# SIMPLE enhanced email writer

def write_email(description):
    """Write email with optional research"""
    
    # AI can use web_search if it wants
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Write professional emails. Research if needed."},
            {"role": "user", "content": f"Write email: {description}"}
        ],
        tools=[web_search_tool]
    )
    
    # If AI used web_search, we'd execute it here
    # For now, just return the email
    return response.choices[0].message.content

# Test
email = write_email("Inform team about new AI features")
print(email)

None


## 📚 COMPLETE VERSION - Enhanced Email Writer

In [9]:
class EnhancedEmailWriter:
    """
    COMPLETE email writer that can research topics.
    """
    
    def __init__(self):
        self.tools = [web_search_tool]
        self.functions = {"web_search": web_search}
    
    def write(self, description, tone="professional"):
        print(f"\n📧 Writing email: {description}")
        print(f"   Tone: {tone}\n")
        
        system_prompt = f"""You are a professional email writer.
        Write in a {tone} tone.
        If you need current information, use web_search.
        Include subject, greeting, body, closing."""
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Write email: {description}"}
        ]
        
        # First API call
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            tools=self.tools
        )
        
        response_message = response.choices[0].message
        
        # Check if AI wants to research
        if response_message.tool_calls:
            messages.append(response_message)
            
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"🔍 Researching: {function_args.get('query', 'N/A')}")
                
                result = self.functions[function_name](**function_args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
            
            # Get final email with research
            final_response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages
            )
            
            return final_response.choices[0].message.content
        
        return response_message.content

# Test
writer = EnhancedEmailWriter()
email = writer.write("Announce new AI features to team")

print("\n" + "=" * 70)
print("📧 EMAIL")
print("=" * 70)
print(email)
print("=" * 70)


📧 Writing email: Announce new AI features to team
   Tone: professional


📧 EMAIL
## Subject: Exciting Announcement: New AI Features Implemented!

Dear Team,

I am thrilled to announce that we have successfully implemented new Artificial Intelligence (AI) features in our system. These enhancements are designed to streamline processes, improve efficiency, and enhance the overall user experience.

The new AI features include advanced data analytics, predictive modeling, and automated decision-making capabilities. With these additions, we can expect faster insights, more accurate forecasts, and smarter recommendations to drive our projects forward.

I encourage everyone to explore these new AI capabilities and familiarize themselves with the enhanced functionalities. Training sessions will be scheduled to provide guidance on how to leverage these features effectively.

Thank you for your dedication and hard work in embracing innovation to improve our workflows. I am confident that these 

---
# Part 2: Smart Summarizer (with Analytics)

## 🎯 SIMPLE VERSION

In [10]:
# SIMPLE smart summarizer

def summarize_smart(text):
    """Summarize with basic analytics"""
    
    # Count words
    words = len(text.split())
    
    # Get summary
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Summarize in 2-3 sentences"},
            {"role": "user", "content": f"Summarize: {text}"}
        ]
    )
    
    summary = response.choices[0].message.content
    summary_words = len(summary.split())
    
    print(f"Original: {words} words")
    print(f"Summary: {summary_words} words")
    print(f"Reduction: {((words - summary_words) / words * 100):.1f}%\n")
    print(f"Summary: {summary}")

# Test
sample = """Artificial intelligence is transforming industries.
Machine learning enables computers to learn from data.
Natural language processing helps computers understand human language.
The adoption of AI is accelerating across all sectors."""

summarize_smart(sample)

Original: 30 words
Summary: 47 words
Reduction: -56.7%

Summary: Artificial intelligence is revolutionizing various industries through machine learning and natural language processing technologies. Machine learning allows computers to learn from data, while natural language processing helps them understand human language. The widespread adoption of AI is rapidly increasing across all sectors due to its transformative capabilities.


## 📚 COMPLETE VERSION - Smart Summarizer

In [11]:
class SmartSummarizer:
    """
    COMPLETE summarizer with detailed analytics.
    """
    
    def summarize(self, text, style="short"):
        print(f"\n📝 Summarizing ({style} style)...\n")
        
        # Analytics
        word_count = len(text.split())
        sentence_count = text.count('.') + text.count('!') + text.count('?')
        
        # Style instructions
        styles = {
            "short": "1-2 sentences",
            "medium": "A paragraph (3-4 sentences)",
            "detailed": "Multiple paragraphs"
        }
        
        # Get summary
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": f"Summarize as {styles.get(style, styles['short'])}"},
                {"role": "user", "content": f"Summarize:\n{text}"}
            ],
            temperature=0.3
        )
        
        summary = response.choices[0].message.content
        summary_words = len(summary.split())
        reduction = ((word_count - summary_words) / word_count * 100)
        
        # Display
        print("=" * 70)
        print("📊 SUMMARY ANALYSIS")
        print("=" * 70)
        print(f"Original: {word_count} words, ~{sentence_count} sentences")
        print(f"Summary: {summary_words} words")
        print(f"Reduction: {reduction:.1f}%")
        print(f"Style: {style.title()}")
        print()
        print("-" * 70)
        print("SUMMARY")
        print("-" * 70)
        print(summary)
        print("-" * 70)

# Test
summarizer = SmartSummarizer()

sample_text = """Artificial intelligence is transforming how businesses operate.
Companies are using AI for customer service, data analysis, and automation.
Machine learning models identify patterns in large datasets.
Natural language processing enables computers to understand human language.
AI adoption is accelerating across all industries and business sizes.
Challenges include data privacy, ethics, and finding skilled professionals."""

summarizer.summarize(sample_text, style="short")


📝 Summarizing (short style)...

📊 SUMMARY ANALYSIS
Original: 54 words, ~6 sentences
Summary: 49 words
Reduction: 9.3%
Style: Short

----------------------------------------------------------------------
SUMMARY
----------------------------------------------------------------------
Artificial intelligence is revolutionizing business operations by enabling companies to utilize AI for customer service, data analysis, automation, and more. This technology, including machine learning and natural language processing, is being increasingly adopted across various industries and businesses, although challenges such as data privacy, ethics, and talent acquisition persist.
----------------------------------------------------------------------


---
# Part 3: Complete Multi-Capability Assistant

## 📚 COMPLETE VERSION - Your Project Template!

In [12]:
class MultiCapabilityAssistant:
    """
    COMPLETE multi-capability assistant.
    
    Features:
    - Chat with memory
    - Enhanced email writer (with research)
    - Smart summarizer (with analytics)
    - Calculator, web search, data analysis tools
    
    This is your PROJECT TEMPLATE - customize it!
    """
    
    def __init__(self):
        # All tools
        self.tools = [calculator_tool, web_search_tool, data_analyzer_tool]
        self.functions = {
            "calculate": calculate,
            "web_search": web_search,
            "analyze_data": analyze_data
        }
        
        # Sub-components
        self.email_writer = EnhancedEmailWriter()
        self.summarizer = SmartSummarizer()
        
        print("✅ Multi-Capability Assistant initialized!")
        print("   Capabilities: Chat, Email, Summarize, Calculate, Search, Analyze\n")
    
    def route_request(self, request):
        """Decide which capability to use"""
        request_lower = request.lower()
        
        if any(word in request_lower for word in ['email', 'write to', 'compose']):
            return 'email'
        elif any(word in request_lower for word in ['summarize', 'summary']):
            return 'summarize'
        elif any(word in request_lower for word in ['calculate', 'math', 'average']):
            return 'tools'
        else:
            return 'general'
    
    def process(self, request):
        """Process any request intelligently"""
        print(f"\n{'=' * 70}")
        print(f"📥 Request: {request}")
        print("=" * 70)
        
        capability = self.route_request(request)
        print(f"🎯 Using: {capability.upper()}\n")
        
        if capability == 'email':
            return self.email_writer.write(request)
        elif capability == 'summarize':
            return "Please provide text to summarize."
        elif capability == 'tools':
            return self.use_tools(request)
        else:
            return self.general_assistant(request)
    
    def use_tools(self, query):
        """Use tools to answer query"""
        messages = [{"role": "user", "content": query}]
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            tools=self.tools
        )
        
        response_message = response.choices[0].message
        
        if not response_message.tool_calls:
            return response_message.content
        
        messages.append(response_message)
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"🔧 Using {function_name}...")
            
            result = self.functions[function_name](**function_args)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        
        final_response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        
        return final_response.choices[0].message.content
    
    def general_assistant(self, query):
        """General purpose assistant"""
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": query}],
            tools=self.tools
        )
        return response.choices[0].message.content

# Create the assistant
assistant = MultiCapabilityAssistant()

✅ Multi-Capability Assistant initialized!
   Capabilities: Chat, Email, Summarize, Calculate, Search, Analyze



In [13]:
# Test the complete assistant!

tests = [
    "Write an email to my team about our Q2 results",
    "What's 25% of 160?",
    "What's the weather like?"
]

for test in tests:
    result = assistant.process(test)
    print(f"\n💬 Response:\n{result}\n")
    print("=" * 70)


📥 Request: Write an email to my team about our Q2 results
🎯 Using: EMAIL


📧 Writing email: Write an email to my team about our Q2 results
   Tone: professional


💬 Response:
## Subject:
Update on Q2 Results

## Greeting:
Dear Team,

## Body:
I hope this message finds you well. I am pleased to share with you the update on our Q2 results. Our team has worked diligently to achieve our goals and I am proud to announce that we have surpassed our targets for the quarter.

The Q2 results reflect the collective effort and dedication of each team member. Your hard work and commitment have contributed significantly to our success. I want to express my gratitude to each one of you for your valuable contributions.

I will be scheduling a meeting next week to discuss the details of the Q2 results and to celebrate our achievements as a team. Please mark your calendars for this important meeting.

Thank you once again for your exceptional work during Q2. Let's continue this momentum as we move forw

---
# Your Project: Customize This!

## Requirements:

### 1. Three Capabilities (Minimum)
- ✅ Enhanced Email Writer (with research)
- ✅ Smart Summarizer (with analytics)
- ✅ Chat OR Data Analyzer OR Your Choice

### 2. Four+ Tools (Minimum)
- ✅ Calculator
- ✅ Web search
- ✅ Data analyzer
- 💡 Add your own! (weather, time, converter, etc.)

### 3. Good Design
- ✅ Input validation
- ✅ Error handling
- ✅ Clear code structure
- ✅ Documentation

## Ideas to Add:

- **Weather tool** - Get current weather
- **Time tool** - Get time in different zones
- **Unit converter** - Convert units
- **File reader** - Read and process files
- **Task manager** - Manage to-do lists
- **Your idea!**

## Submission:

- GitHub repository
- README with setup instructions
- All code working and tested
- Example usage/screenshots

**Due:** Before Session 3 (Next Saturday)

---
# Summary

## ✅ What You Built:

1. **Enhanced Email Writer** - With web research
2. **Smart Summarizer** - With text analytics
3. **Multi-Capability Assistant** - Complete project template

## 🎯 Complete Skill Set:

**Session 1 + Session 2 Combined:**
- ✅ Chat with conversation memory
- ✅ Text summarization
- ✅ Email generation
- ✅ Function calling & tools
- ✅ Calculator, search, data analysis
- ✅ Multi-capability routing
- ✅ Error handling & validation
